# Module 2 - Sous-performance territoriale

Objectif : comparer les ventes mensuelles d'une ville avec son potentiel démographique.

La population est seulement une proxy. Les quadrants servent à prioriser les villes, pas à prouver une sous-couverture commerciale.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.territory_analysis import (
    aggregate_regions,
    build_city_table,
    classify_territories,
    top_opportunities,
)

INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'transactions_normalized.csv'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'generated'

## 1. Charger et contrôler la couverture

Les villes sans population ou sans région canonique seront exclues du modèle territorial, mais comptées dans le contrôle de couverture.

In [ ]:
transactions = pd.read_csv(INPUT_PATH, parse_dates=['Date'])
print(f'Lignes : {len(transactions):,}')
print(transactions.groupby('Country').size())
print(f'Population manquante : {transactions["Population"].isna().sum():,}')

## 2. Agréger au niveau ville

La performance est calculée comme `Sales / ActiveMonths`, afin de comparer les fenêtres Allemagne et Pologne.

In [ ]:
city = build_city_table(transactions)
city[['City', 'Country', 'Population', 'Sales', 'MonthlySales', 'RegionCanonical']].head()

## 3. Modèle population-performance

On ajuste une régression linéaire sur les logarithmes. Le résidu mesure si une ville vend plus ou moins que ce que sa population laisse attendre.

In [ ]:
classified, model, population_median = classify_territories(city)
print(f'Villes utilisées : {len(classified):,}')
print(f'Médiane de population : {population_median:,.0f}')
predicted_log_sales = model.predict(np.log(classified[['Population']].to_numpy()))
actual_log_sales = np.log(classified['MonthlySales'])
r_squared = 1 - ((actual_log_sales - predicted_log_sales) ** 2).sum() / ((actual_log_sales - actual_log_sales.mean()) ** 2).sum()
print(f'R² : {r_squared:.3f}')
classified['Quadrant'].value_counts()

## 4. Préparer les tables Power BI

Les trois sorties sont au niveau ville, région et opportunité. Elles seront régénérées à chaque exécution.

In [ ]:
regions = aggregate_regions(classified)
opportunities = top_opportunities(classified)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
classified.to_csv(OUTPUT_DIR / 'territories_by_city.csv', index=False)
regions.to_csv(OUTPUT_DIR / 'territories_by_region.csv', index=False)
opportunities.to_csv(OUTPUT_DIR / 'territory_opportunities.csv', index=False)
opportunities.head(10)